# Experiment 1 (continued) — resume the matched-scale run

The first session hit Kaggle's 12-hour cap and was killed with exit code 137.
Nothing is lost. Checkpoints are written to
`/kaggle/working/project/aicd/artifacts/`, which is inside the notebook output,
so the epoch-1 checkpoint survived along with the built corpus.

This notebook picks up at epoch 2 and runs the evaluation. It needs about
**7 hours**, comfortably inside one session: roughly 4.9 h for the last epoch
and 2 h for evaluation.

Only two files matter for resuming:

| File | Why |
|---|---|
| `artifacts/branch_a_matched_ckpt.pt` | model, optimiser, scheduler, scaler, epoch |
| `artifacts/data/splits.parquet` | the training data |

The 647 MB of raw DroidCollection parquets are **not** needed, so nothing is
downloaded again.

## Settings

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` (the tokenizer and base weights still come from HuggingFace) |
| **Persistence** | `Files only` |

**Input → Add Input**, and add *both*:
1. **Datasets →** `aicd-code`
2. **Your Work / Notebooks →** the `Matched-scale` notebook, version 3

Then **Save Version → Save & Run All (Commit)**.

In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a pipeline stage and stop the notebook if it fails.

    Without the raise a failed stage prints a traceback and the next cell
    happily trains on whatever stale data is lying around.
    """
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Add your code dataset: right panel -> Input -> Add Input -> Datasets,\n"
        "then search for the dataset you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
print("configs present:", sorted(p.name for p in (dest/"configs").glob("kaggle*.yaml")))

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Restore the checkpoint and the corpus

Searches every attached input for the two files that matter. If the checkpoint
is missing this stops immediately rather than silently starting a fresh 15-hour
run, which is the one failure mode that would waste another whole quota.

In [ ]:
art = WORK / "aicd" / "artifacts"
(art / "data").mkdir(parents=True, exist_ok=True)

def newest(pattern):
    hits = sorted(pathlib.Path("/kaggle/input").rglob(pattern),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0] if hits else None

ck_src = newest("branch_a_matched_ckpt.pt")
sp_src = newest("splits.parquet")

if ck_src is None:
    raise SystemExit(
        "branch_a_matched_ckpt.pt not found in any attached input.\n"
        "Add the previous run's output: Input -> Add Input -> Your Work ->\n"
        "the 'Matched-scale' notebook. Without it this notebook would start\n"
        "training from scratch and burn another 15 hours.")
if sp_src is None:
    raise SystemExit("splits.parquet not found. Attach the same notebook output.")

shutil.copy(ck_src, art / "branch_a_matched_ckpt.pt")
shutil.copy(sp_src, art / "data" / "splits.parquet")
print(f"checkpoint <- {ck_src}  ({ck_src.stat().st_size/1e9:.2f} GB)")
print(f"splits     <- {sp_src}  ({sp_src.stat().st_size/1e6:.0f} MB)")

# Read the epoch out of the checkpoint before training, so the plan is visible
# up front rather than inferred from the log an hour later.
ck = torch.load(art / "branch_a_matched_ckpt.pt", map_location="cpu",
                weights_only=False)
done = ck["epoch"] + 1
print(f"\ncompleted epochs : {done} of 3")
print(f"this session runs: epoch {done} to 2  ({3 - done} epoch(s))")
print(f"estimated        : {(3-done)*4.9:.1f} h training + ~2 h evaluation")
del ck
elapsed("restored")

## 4. Finish training, then evaluate

Same command as the first session. `--resume` finds the checkpoint and starts
at the next epoch; `--tag matched` keeps this run's artefacts separate from the
original single-shard run.

In [ ]:
run(["-m", "aicd.models.modernbert_triplet", "--config", "kaggle_matched.yaml",
     "--tag", "matched", "--resume"])
elapsed("branch A (matched scale) finished")

## 5. The verdict

In [ ]:
r = json.load(open(WORK / "aicd" / "eval" / "reports" / "branch_a_matched.json"))["slices"]
ORIG = {"s1_in_distribution": 0.8977, "s2_unseen_generator": 0.8685,
        "s3_unseen_language": 0.5667, "s4_unseen_domain": 0.4029,
        "s5_compound": 0.2378}
print(f"{'condition':24s} {'matched':>9s} {'original':>9s}   n rows")
print("-" * 56)
for s, o in ORIG.items():
    if s in r:
        print(f"{s:24s} {r[s]['macro_f1']:9.4f} {o:9.4f}   {r[s].get('n', 0):,}")
s1, s5 = r["s1_in_distribution"]["macro_f1"], r["s5_compound"]["macro_f1"]
print(f"\nmatched  S1 -> S5: {s1:.4f} -> {s5:.4f}   drop {s1-s5:.4f}")
print(f"original S1 -> S5: 0.8977 -> 0.2378   drop {0.8977-0.2378:.4f}")
print("\ntrained on 394,624 rows against the original 196,854 (2.0x)\n")
if s5 < 0.45:
    print("COLLAPSE PERSISTS at twice the training data.")
    print("Training-set size is excluded as the explanation, and the causal")
    print("claim in the paper stands. This closes the confound in Section VIII.")
else:
    print("COLLAPSE DOES NOT PERSIST. Report this: part of the original effect")
    print("was a data-volume artefact and the framing must change. The exposure")
    print("audit and contamination results are unaffected either way.")

## 6. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

# The probability arrays are what the analysis modules re-read at home, and
# they are small. The model weights are hundreds of MB and are not needed to
# reproduce any number in the paper, so they stay behind.
art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for f in art.glob("labels.parquet"):
    shutil.copy(f, npy / f.name)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (download this from the Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")